In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Dataset URL from the instructions
url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'
df = pd.read_csv(url)

# Select only the required columns
selected_columns = [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year',
    'fuel_efficiency_mpg'
]

df = df[selected_columns]
df.head()

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,2180,243.0,3870,2006,31.9
1,2390,272.0,4210,2008,31.3
2,2320,267.0,4240,1996,27.5
3,2130,258.0,4490,1989,28.5
4,2580,304.0,4510,1994,31.0


In [3]:
# Question 1: Check missing values across the selected columns
missing_counts = df.isnull().sum()
print("Missing values count per column:")
print(missing_counts)

# Identify the column with missing values
missing_col = missing_counts[missing_counts > 0].index.tolist()
print("\nColumn with missing values:", missing_col)

Missing values count per column:
engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

Column with missing values: ['horsepower']


In [4]:
# Question 2: Median (50th percentile) for 'horsepower'
median_hp = df['horsepower'].median()
print(f"Median horsepower: {median_hp}")

Median horsepower: 254.0


In [7]:
# Determine split sizes
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

# Shuffle indices with seed 42
np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

# Create train, validation, and test splits
df_train = df.iloc[idx[:n_train]].reset_index(drop=True)
df_val = df.iloc[idx[n_train:n_train + n_val]].reset_index(drop=True)
df_test = df.iloc[idx[n_train + n_val:]].reset_index(drop=True)

# Separate target variable (fuel_efficiency_mpg)
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

# Remove the target column from the feature sets
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [9]:
def train_linear_regression(X, y):
    # Add bias term (column of 1s)
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    # Normal equation: (X^T * X)^(-1) * X^T * y
    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)

    return w[0], w[1:]

def rmse(y, y_pred):
    error = y_pred - y
    mse = (error ** 2).mean()
    return np.sqrt(mse)

In [10]:
# Helper to prepare X with a given fill value
def prepare_X(df_subset, fill_value):
    df_num = df_subset.copy()
    df_num = df_num.fillna(fill_value)
    return df_num.values

# --- Option 1: Fill with 0 ---
X_train_zero = prepare_X(df_train, fill_value=0)
w_0_zero, w_zero = train_linear_regression(X_train_zero, y_train)

X_val_zero = prepare_X(df_val, fill_value=0)
y_pred_zero = w_0_zero + X_val_zero.dot(w_zero)
score_zero = round(rmse(y_val, y_pred_zero), 3)

# --- Option 2: Fill with Mean of Training set ---
# Compute mean strictly from the training data
mean_val = df_train.mean()

X_train_mean = prepare_X(df_train, fill_value=mean_val)
w_0_mean, w_mean = train_linear_regression(X_train_mean, y_train)

X_val_mean = prepare_X(df_val, fill_value=mean_val)
y_pred_mean = w_0_mean + X_val_mean.dot(w_mean)
score_mean = round(rmse(y_val, y_pred_mean), 3)

# Compare results
print(f"RMSE with 0:    {score_zero}")
print(f"RMSE with mean: {score_mean}")

RMSE with 0:    2.205
RMSE with mean: 2.202


In [12]:
def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    reg = r * np.eye(XTX.shape[0])
    XTX = XTX + reg

    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)

    return w[0], w[1:]

In [13]:
# Prepare feature matrices filled with 0
X_train = prepare_X(df_train, fill_value=0)
X_val = prepare_X(df_val, fill_value=0)

r_values = [0, 0.01, 0.1, 1, 5, 10, 100]

print("r\t\tRMSE")
print("-" * 20)

best_r = None
best_rmse = float('inf')

for r in r_values:
    w_0, w = train_linear_regression_reg(X_train, y_train, r=r)
    y_pred = w_0 + X_val.dot(w)
    score = round(rmse(y_val, y_pred), 4)
    
    print(f"{r:<8}\t{score}")
    
    if score < best_rmse:
        best_rmse = score
        best_r = r

print("-" * 20)
print(f"Best r: {best_r} with RMSE: {best_rmse}")

r		RMSE
--------------------
0       	2.2053
0.01    	2.2058
0.1     	2.2241
1       	2.3492
5       	2.4094
10      	2.4195
100     	2.4292
--------------------
Best r: 0 with RMSE: 2.2053


In [14]:
# Question 5: Evaluating stability across different seeds
seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
scores = []

n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

for s in seeds:
    # 1. Shuffle indices with the current seed
    np.random.seed(s)
    idx = np.arange(n)
    np.random.shuffle(idx)

    # 2. Split into train, validation, and test sets
    df_train_s = df.iloc[idx[:n_train]].reset_index(drop=True)
    df_val_s = df.iloc[idx[n_train:n_train + n_val]].reset_index(drop=True)

    # 3. Extract the target variable
    y_train_s = df_train_s['fuel_efficiency_mpg'].values
    y_val_s = df_val_s['fuel_efficiency_mpg'].values

    # 4. Remove target from features
    del df_train_s['fuel_efficiency_mpg']
    del df_val_s['fuel_efficiency_mpg']

    # 5. Fill missing values with 0
    X_train_s = prepare_X(df_train_s, fill_value=0)
    X_val_s = prepare_X(df_val_s, fill_value=0)

    # 6. Train model without regularization
    w_0, w = train_linear_regression(X_train_s, y_train_s)

    # 7. Evaluate on validation set
    y_pred_s = w_0 + X_val_s.dot(w)
    score = rmse(y_val_s, y_pred_s)
    scores.append(score)

# Compute and round standard deviation to 3 decimal digits
std_val = round(float(np.std(scores)), 3)

print("Validation RMSE scores:", [round(s, 4) for s in scores])
print(f"Standard deviation: {std_val}")

Validation RMSE scores: [np.float64(2.2392), np.float64(2.2021), np.float64(2.1634), np.float64(2.2044), np.float64(2.2019), np.float64(2.2458), np.float64(2.2697), np.float64(2.1908), np.float64(2.2166), np.float64(2.207)]
Standard deviation: 0.029


In [15]:
# Question 6: Train on combined train + val with seed 9 and r=0.001

# 1. Split the dataset using seed 9
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(9)
idx = np.arange(n)
np.random.shuffle(idx)

df_train_q6 = df.iloc[idx[:n_train]].reset_index(drop=True)
df_val_q6 = df.iloc[idx[n_train:n_train + n_val]].reset_index(drop=True)
df_test_q6 = df.iloc[idx[n_train + n_val:]].reset_index(drop=True)

# 2. Combine train and validation datasets
df_full_train = pd.concat([df_train_q6, df_val_q6]).reset_index(drop=True)

# Extract targets
y_full_train = df_full_train['fuel_efficiency_mpg'].values
y_test = df_test_q6['fuel_efficiency_mpg'].values

# Remove target from feature matrices
del df_full_train['fuel_efficiency_mpg']
del df_test_q6['fuel_efficiency_mpg']

# 3. Fill missing values with 0
X_full_train = prepare_X(df_full_train, fill_value=0)
X_test = prepare_X(df_test_q6, fill_value=0)

# 4. Train regularized linear regression with r = 0.001
w_0, w = train_linear_regression_reg(X_full_train, y_full_train, r=0.001)

# 5. Predict on test dataset and calculate RMSE
y_pred_test = w_0 + X_test.dot(w)
test_rmse = round(rmse(y_test, y_pred_test), 3)

print(f"Test RMSE: {test_rmse}")

Test RMSE: 2.236
